In [1]:
import scanpy as sc
import pandas as pd
import numpy as np
import gc

##### E-MTAB-8142_Reynolds_et_al

In [ ]:
adata = sc.read_h5ad("/nfs/cellgeni/pasham/projects/2110.skin.visium/rds/v2/rey.rep/raw.as.rey.gene.h5ad")

In [ ]:
adata

In [ ]:
adata.obs[["Site", "Status"]].value_counts()

In [31]:
adata.obs.Site.value_counts()

non_lesion    317005
lesion        134589
Name: Site, dtype: int64

In [25]:
adata.obs.rename(columns = {"donor":"donor_id", "stage":"development_stage", "Status":"disease", "Tissue":"tissue",
                           "Sex":"sex", "Age":"age"}, inplace=True)

In [69]:
adata.obs.development_stage.replace("adult", "Adult", inplace=True)

In [27]:
adata.obs["dataset_id"] = "E-MTAB-8142_Reynolds_2021"

In [70]:
adata.obs.development_stage.value_counts()

Adult    451594
Name: development_stage, dtype: int64

In [63]:
for i in range(adata.n_obs):
    if (adata.obs.loc[adata.obs.index[i], "disease"] != "Healthy"):
        adata.obs.loc[adata.obs.index[i], "cell_source"] = adata.obs.tissue[i]+"_"+adata.obs.Site[i]
    else:
        adata.obs.loc[adata.obs.index[i], "cell_source"] = adata.obs.tissue[i]

In [64]:
adata.obs.cell_source.value_counts()

Epidermis               99508
Dermis                  96231
Dermis_lesion           73099
Dermis_non_lesion       62901
Epidermis_lesion        61490
Epidermis_non_lesion    58365
Name: cell_source, dtype: int64

In [71]:
s_path = "/nfs/team205/bh14/Datasets/Remapped/raw_adata/"
adata.write_h5ad(s_path+"Tissue_or_Nonimmune/E-MTAB-8142_Reynolds_2021.h5ad")

In [4]:
# Down-sample Reynolds dataset after QC
# Select only immune cell, fibroblast and endothelial cells from author annotation

In [2]:
temp = sc.read("/nfs/team205/bh14/Datasets/Remapped/raw_adata/Reynolds_old.h5ad")

In [2]:
path = "/nfs/team205/bh14/Datasets/Remapped/raw_adata/QC_Doublet/Tissue_or_Nonimmune/"

In [4]:
adata = sc.read(path+"E-MTAB-8142_Reynolds_2021.h5ad")

In [5]:
adata.obs

,barcode,sample_id,batch,disease,Site,tissue,Enrichment,Location,sex,age,...,total_counts_rb,pct_counts_rb,donor_id,log1p_total_counts,dataset_id,cell_source,doublet_scores,predicted_doublets,QC,author_annotation
AAACCTGAGCTACCGC-1-SKN8090524,AAACCTGAGCTACCGC,SKN8090524,31,Eczema,non_lesion,Epidermis,Live_single_spike,Lower_back,Female,62.0,...,0.0,0.0,E2,7.303170,E-MTAB-8142_Reynolds_2021,Epidermis_non_lesion,0.046689,False,Pass,Undifferentiated_KC*
AAACCTGCACCTCGGA-1-SKN8090524,AAACCTGCACCTCGGA,SKN8090524,31,Eczema,non_lesion,Epidermis,Live_single_spike,Lower_back,Female,62.0,...,0.0,0.0,E2,7.737616,E-MTAB-8142_Reynolds_2021,Epidermis_non_lesion,0.036046,False,Pass,Undifferentiated_KC*
AAACCTGGTCAGAAGC-1-SKN8090524,AAACCTGGTCAGAAGC,SKN8090524,31,Eczema,non_lesion,Epidermis,Live_single_spike,Lower_back,Female,62.0,...,0.0,0.0,E2,8.372630,E-MTAB-8142_Reynolds_2021,Epidermis_non_lesion,0.051362,False,Pass,Undifferentiated_KC*
AAACCTGGTCGAACAG-1-SKN8090524,AAACCTGGTCGAACAG,SKN8090524,31,Eczema,non_lesion,Epidermis,Live_single_spike,Lower_back,Female,62.0,...,0.0,0.0,E2,8.679312,E-MTAB-8142_Reynolds_2021,Epidermis_non_lesion,0.016450,False,Pass,Undifferentiated_KC*
AAACCTGTCTCCGGTT-1-SKN8090524,AAACCTGTCTCCGGTT,SKN8090524,31,Eczema,non_lesion,Epidermis,Live_single_spike,Lower_back,Female,62.0,...,0.0,0.0,E2,8.359838,E-MTAB-8142_Reynolds_2021,Epidermis_non_lesion,0.029704,False,Pass,Undifferentiated_KC*
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTTGGTTTCAGGCCCA-1-4820STDY7389014,TTTGGTTTCAGGCCCA,4820STDY7389014,64,Healthy,non_lesion,Epidermis,Lymphocytes,Breast,Female,NaN,...,0.0,0.0,s3,6.997596,E-MTAB-8142_Reynolds_2021,Epidermis,0.009379,False,Low_nFeature,LC_2
TTTGGTTTCGCCTGTT-1-4820STDY7389014,TTTGGTTTCGCCTGTT,4820STDY7389014,64,Healthy,non_lesion,Epidermis,Lymphocytes,Breast,Female,NaN,...,0.0,0.0,s3,8.426612,E-MTAB-8142_Reynolds_2021,Epidermis,0.031253,False,Pass,Th
TTTGTCAAGGAATCGC-1-4820STDY7389014,TTTGTCAAGGAATCGC,4820STDY7389014,64,Healthy,non_lesion,Epidermis,Lymphocytes,Breast,Female,NaN,...,0.0,0.0,s3,7.913887,E-MTAB-8142_Reynolds_2021,Epidermis,0.011323,False,Pass,Th
TTTGTCAAGGACTGGT-1-4820STDY7389014,TTTGTCAAGGACTGGT,4820STDY7389014,64,Healthy,non_lesion,Epidermis,Lymphocytes,Breast,Female,NaN,...,0.0,0.0,s3,8.163371,E-MTAB-8142_Reynolds_2021,Epidermis,0.008722,False,Pass,Th


In [6]:
common_obs = [obs for obs in temp.obs_names if obs in adata.obs_names]

In [7]:
len(common_obs)

451594

In [8]:
temp.shape

(539962, 33538)

In [9]:
temp = temp[common_obs]

In [10]:
temp.shape

(451594, 33538)

In [11]:
temp.obs.final_clustering

index
AAACCTGAGCTACCGC-1-SKN8090524         Undifferentiated_KC*
AAACCTGCACCTCGGA-1-SKN8090524         Undifferentiated_KC*
AAACCTGGTCAGAAGC-1-SKN8090524         Undifferentiated_KC*
AAACCTGGTCGAACAG-1-SKN8090524         Undifferentiated_KC*
AAACCTGTCTCCGGTT-1-SKN8090524         Undifferentiated_KC*
                                              ...         
TTTGGTTTCAGGCCCA-1-4820STDY7389014                    LC_2
TTTGGTTTCGCCTGTT-1-4820STDY7389014                      Th
TTTGTCAAGGAATCGC-1-4820STDY7389014                      Th
TTTGTCAAGGACTGGT-1-4820STDY7389014                      Th
TTTGTCACACTACAGT-1-4820STDY7389014                      Th
Name: final_clustering, Length: 451594, dtype: category
Categories (40, object): ['DC1', 'DC2', 'Differentiated_KC', 'Differentiated_KC*', ..., 'VE3', 'moDC_1', 'moDC_2', 'moDC_3']

In [12]:
adata.obs["author_annotation"] = temp.obs.final_clustering

In [13]:
np.unique(adata.obs.author_annotation)

array(['DC1', 'DC2', 'Differentiated_KC', 'Differentiated_KC*', 'F1',
       'F2', 'F3', 'ILC1_3', 'ILC1_NK', 'ILC2', 'Inf_mono', 'LC_1',
       'LC_2', 'LC_3', 'LC_4', 'LE1', 'LE2', 'Macro_1', 'Macro_2',
       'Mast_cell', 'Melanocyte', 'MigDC', 'Mono', 'NK',
       'Pericyte_1_non_inflamm', 'Pericyte_2_inflamm', 'Plasma',
       'Proliferating_KC', 'Schwann1', 'Schwann2', 'Tc', 'Th', 'Treg',
       'Undifferentiated_KC*', 'VE1', 'VE2', 'VE3', 'moDC_1', 'moDC_2',
       'moDC_3'], dtype=object)

In [14]:
adata2 = adata[adata.obs.author_annotation.isin(['DC1', 'DC2', 'F1',
       'F2', 'F3', 'ILC1_3', 'ILC1_NK', 'ILC2', 'Inf_mono', 'LC_1',
       'LC_2', 'LC_3', 'LC_4', 'LE1', 'LE2', 'Macro_1', 'Macro_2',
       'Mast_cell', 'MigDC', 'Mono', 'NK', 'Plasma', 'Tc', 'Th', 'Treg',
       'VE1', 'VE2', 'VE3', 'moDC_1', 'moDC_2','moDC_3'])]

In [15]:
adata2

View of AnnData object with n_obs × n_vars = 286427 × 36601
    obs: 'barcode', 'sample_id', 'batch', 'disease', 'Site', 'tissue', 'Enrichment', 'Location', 'sex', 'age', 'development_stage', 'full_clustering', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'total_counts_rb', 'pct_counts_rb', 'donor_id', 'log1p_total_counts', 'dataset_id', 'cell_source', 'doublet_scores', 'predicted_doublets', 'QC', 'author_annotation'
    var: 'gene_name', 'mt', 'rb', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts'

In [16]:
adata2.var_names = adata2.var.gene_name

In [17]:
adata2.var

,gene_name,mt,rb,n_cells_by_counts,mean_counts,pct_dropout_by_counts,total_counts
gene_name,,,,,,,
MIR1302-2HG,MIR1302-2HG,False,False,6,0.000013,99.998671,6.0
FAM138A,FAM138A,False,False,0,0.000000,100.000000,0.0
OR4F5,OR4F5,False,False,0,0.000000,100.000000,0.0
AL627309.1,AL627309.1,False,False,366,0.000810,99.918954,366.0
AL627309.3,AL627309.3,False,False,4,0.000009,99.999114,4.0
...,...,...,...,...,...,...,...
AC141272.1,AC141272.1,False,False,0,0.000000,100.000000,0.0
AC023491.2,AC023491.2,False,False,3,0.000007,99.999336,3.0
AC007325.1,AC007325.1,False,False,29,0.000064,99.993578,29.0


In [ ]:
#Divide in two datasets

In [3]:
#adata2 = sc.read(path+"E-MTAB-8142_Reynolds_2021.h5ad")

In [10]:
dataset = adata2.obs.dataset_id.astype(str)

In [12]:
tissue = adata2.obs.tissue.astype(str)

In [14]:
dist_dataset = dataset+"_"+tissue

In [15]:
dist_dataset

AAAGATGTCACGGTTA-1-SKN8090524         E-MTAB-8142_Reynolds_2021_Epidermis
AAAGCAAGTTGTCGCG-1-SKN8090524         E-MTAB-8142_Reynolds_2021_Epidermis
AAAGTAGAGTACGACG-1-SKN8090524         E-MTAB-8142_Reynolds_2021_Epidermis
AAAGTAGGTACCGAGA-1-SKN8090524         E-MTAB-8142_Reynolds_2021_Epidermis
AAAGTAGGTTCAGGCC-1-SKN8090524         E-MTAB-8142_Reynolds_2021_Epidermis
                                                     ...                 
TTTGGTTTCAGGCCCA-1-4820STDY7389014    E-MTAB-8142_Reynolds_2021_Epidermis
TTTGGTTTCGCCTGTT-1-4820STDY7389014    E-MTAB-8142_Reynolds_2021_Epidermis
TTTGTCAAGGAATCGC-1-4820STDY7389014    E-MTAB-8142_Reynolds_2021_Epidermis
TTTGTCAAGGACTGGT-1-4820STDY7389014    E-MTAB-8142_Reynolds_2021_Epidermis
TTTGTCACACTACAGT-1-4820STDY7389014    E-MTAB-8142_Reynolds_2021_Epidermis
Length: 286427, dtype: object

In [16]:
adata2.obs["dataset_id"] = dist_dataset

In [18]:
adata2.obs.dataset_id.value_counts()

E-MTAB-8142_Reynolds_2021_Dermis       185982
E-MTAB-8142_Reynolds_2021_Epidermis    100445
Name: dataset_id, dtype: int64

In [19]:
adata2.write_h5ad(path+"E-MTAB-8142_Reynolds_2021.h5ad")